# Week 4 — The Graph Lab

The lecture described three things: a database that stores relationships, a language for
following them, and a vector index that finds a starting point in the graph by meaning.
This session runs all three on your own machine and checks each claim the lecture made,
instead of taking it on trust.

**Two graphs.** Parts 2 to 5 use Neo4j's standard **Movies graph**: 38 films, 133 people,
and who acted in, directed, produced, wrote, or reviewed what. It is the dataset of Neo4j's
own documentation and Browser guide, so you will find many more examples of it. Parts 6
to 9 use a graph of **this course**, whose topics carry descriptions written to be
embedded. Everyone works on the same two graphs, so any result can be checked against a
known answer. Your own domain comes in Week 5.

**Before you start.** Neo4j must be running. From the repository root:

```bash
docker compose up -d
```

Then open Neo4j Browser at http://localhost:7474 (user `neo4j`, password `modernai`).
Try the queries in Parts 3 and 4 in Browser as well as here: Browser draws the graph,
and this notebook shows the rows a program receives.

**Cypher version.** The course runs Neo4j 5.26, which speaks Cypher 5. Most of Neo4j's
current online documentation describes Cypher 25, and some of it does not run here. Use the
Cypher 5 manual: https://neo4j.com/docs/cypher-manual/5/introduction/

**On cost.** Parts 6 and 7 call the API. Part 6 embeds the course graph once, about 40
texts, and then one text per question. Part 7 makes three generation calls, against the
separate limit of six per minute, plus one for each question of your own. Part 9 copies
vectors you already paid for in Week 3 and embeds only your questions.

In [ ]:
# Run this notebook from any folder. Week 4's modules live in week4/, and they
# reuse Week 3's embedding client and vector store and Week 2's ask(), so all three
# go on the path.
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "requirements.txt").exists()), pathlib.Path.cwd().parent)
for _d in ("week2", "week3", "week4"):
    sys.path.insert(0, str(_root / _d))

---
## Part 1 — Is Neo4j running?

In [ ]:
from graph_db import connect, run, run_script, counts, breakdown, server_info

driver = connect()
print(server_info(driver))
print(counts(driver))

You should see version `5.26.x` and edition `community`. If `connect()` fails, the
container is not running: run `docker compose up -d` from the repository root and allow
about thirty seconds for it to start.

**One warning before anything else.** `docker compose down -v` deletes the Docker volume,
and your graph with it. `docker compose down` without `-v` is safe. This is why every graph
in this course must be rebuildable from a script, which is the subject of Part 2.

---
## Part 2 — Load the Movies graph, twice

The script is Neo4j's own, copied unchanged into `week4/movies.cypher` from Neo4j's
repository at a pinned commit. (Browser's `:play movies` guide runs the same script, if
you prefer to load it there.) Open `week4/movies.cypher`
after the next cell and read its first twenty lines: uniqueness constraints first, then
every node merged on its key, then relationships merged between nodes that already exist.

In [ ]:
from load_graph import MOVIES_SCRIPT as MOVIES, switch_to    # week4/movies.cypher

switch_to(driver, MOVIES)          # one graph at a time: removes the course graph, if loaded
run_script(driver, MOVIES)
first = breakdown(driver)
print("after one load: ", first)

In [ ]:
run_script(driver, MOVIES)
second = breakdown(driver)
print("after two loads:", second)
print("unchanged:      ", first == second)

The counts did not change. This is **idempotence**: the second load found everything the
first one created, instead of creating it again. You should see 38 `Movie` and 133
`Person` nodes, 171 in all, joined by 253 relationships. Week 5 requires a re-seedable
`ingest.py` for your own graph, and it has to behave the same way.

### Now make the standard mistake, on purpose

`MERGE` matches a **whole pattern**. If any part of the pattern is missing, it creates all
of it, including nodes that already exist. The cell below uses a throwaway label,
`Scratch`, which has no constraint.

In [ ]:
run(driver, "MATCH (n:Scratch) DETACH DELETE n")          # start clean
run(driver, "CREATE (:Scratch {kind: 'person', name: 'Tom Hanks'}), (:Scratch {kind: 'movie', title: 'The Matrix'})")

# WRONG: merges the whole pattern. The relationship does not exist yet, so MERGE
# creates all of it -- two NEW nodes beside the two that were already there.
run(driver, "MERGE (:Scratch {kind: 'person', name: 'Tom Hanks'})-[:ACTED_IN]->(:Scratch {kind: 'movie', title: 'The Matrix'})")

run(driver, "MATCH (n:Scratch) RETURN n.kind AS kind, count(*) AS copies ORDER BY kind")

Two copies of each node. The correct pattern merges each node on its key first, and then
only the relationship between them.

In [ ]:
run(driver, "MATCH (n:Scratch) DETACH DELETE n")
run(driver, "CREATE (:Scratch {kind: 'person', name: 'Tom Hanks'}), (:Scratch {kind: 'movie', title: 'The Matrix'})")

# RIGHT: find (or create) each node on its key, then merge only the relationship.
run(driver, """
    MERGE (p:Scratch {kind: 'person', name: 'Tom Hanks'})
    MERGE (m:Scratch {kind: 'movie', title: 'The Matrix'})
    MERGE (p)-[:ACTED_IN]->(m)
""")

run(driver, "MATCH (n:Scratch) RETURN n.kind AS kind, count(*) AS copies ORDER BY kind")

With a constraint, the same mistake fails loudly instead of silently. `Person.name` and
`Movie.title` are constrained, and Tom Hanks is not in The Matrix, so this whole pattern
does not exist yet:

In [ ]:
try:
    run(driver, "MERGE (:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(:Movie {title: 'The Matrix'})")
except Exception as exc:
    print(type(exc).__name__, "--", str(exc)[:150])

run(driver, "MATCH (n:Scratch) DETACH DELETE n")          # tidy up
print(breakdown(driver) == second)                          # True: the failed statement wrote nothing

---
## Part 3 — Querying by pattern

A Cypher query is a pattern of nodes `( )` and relationships `-[ ]->`, and `MATCH` finds
every place the pattern occurs. Run each query here, then paste it into Neo4j Browser and
switch between the graph view and the table view. **A program receives the table.** In
Week 6 that table becomes text in a model's context.

In [ ]:
# Which films did Tom Hanks act in, oldest first?
run(driver, """
    MATCH (:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)
    RETURN m.title AS title, m.released AS year ORDER BY year
""")

In [ ]:
# Who directed the films Keanu Reeves acted in? Two relationships, one pattern.
run(driver, """
    MATCH (:Person {name: 'Keanu Reeves'})-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d:Person)
    RETURN DISTINCT d.name AS director ORDER BY director
""")

Relationships carry properties too. The role an actor played belongs to neither the actor
nor the film, but to the fact that one acted in the other, so it is stored on the
`ACTED_IN` relationship. In a relational schema it would be a column of the join table.

In [ ]:
run(driver, """
    MATCH (p:Person)-[r:ACTED_IN]->(:Movie {title: 'The Matrix'})
    RETURN p.name AS actor, r.roles AS roles ORDER BY actor
""")

### Aggregation has no `GROUP BY`

Cypher groups by every expression in `RETURN` that is **not** an aggregate. Compare these
two queries.

In [ ]:
# How many films has each actor appeared in?
run(driver, """
    MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
    RETURN p.name AS actor, count(m) AS films ORDER BY films DESC, actor
    LIMIT 5
""")

In [ ]:
# The same question with one extra column. Every count is now 1. Why?
run(driver, """
    MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
    RETURN p.name AS actor, m.title AS title, count(m) AS films ORDER BY films DESC
    LIMIT 5
""")

Adding `m.title` to `RETURN` made it part of the grouping key, so every group now holds a
single film. A reader who expects SQL's `GROUP BY actor` misreads the second query.

`WITH` passes results to the next part of a query. Here it lets the query filter on an
aggregate, which is what `HAVING` does in SQL:

In [ ]:
run(driver, """
    MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
    WITH p, count(m) AS films
    WHERE films >= 5
    RETURN p.name AS actor, films ORDER BY films DESC
""")

In [ ]:
# Who is in the graph but never acted? OPTIONAL MATCH is Cypher's outer join:
# a person with no ACTED_IN relationship is kept, with m = null.
run(driver, """
    MATCH (p:Person)
    OPTIONAL MATCH (p)-[:ACTED_IN]->(m:Movie)
    WITH p, count(m) AS films
    WHERE films = 0
    RETURN p.name AS person ORDER BY person
    LIMIT 10
""")

---
## Part 4 — Traversal, and how fast paths grow

A traversal follows relationships to a depth that is not known in advance. The course
teaches the syntax that conforms to the GQL standard: a **quantified relationship** such as
`-[:ACTED_IN]-{1,6}`, and `SHORTEST`. You will also meet the older forms, `-[:ACTED_IN*1..6]-`
and `shortestPath()`, in examples and in generated code. Both run on Neo4j 5.26.

In [ ]:
# How is Kevin Bacon connected to Meg Ryan? No single fact in the graph says so;
# the answer is a chain of facts -- the kind of question Week 3's retrieval could not answer.
run(driver, """
    MATCH p = SHORTEST 1 (:Person {name: 'Kevin Bacon'})-[]-+(:Person {name: 'Meg Ryan'})
    RETURN [n IN nodes(p) | coalesce(n.name, n.title)] AS route, length(p) AS hops
""")

In [ ]:
# The classic recommendation: actors Tom Hanks has never worked with,
# ranked by how many of his co-actors have.
run(driver, """
    MATCH (tom:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(:Movie)<-[:ACTED_IN]-(co:Person)
          -[:ACTED_IN]->(:Movie)<-[:ACTED_IN]-(suggestion:Person)
    WHERE tom <> suggestion AND NOT (tom)-[:ACTED_IN]->(:Movie)<-[:ACTED_IN]-(suggestion)
    RETURN suggestion.name AS suggestion, count(*) AS strength
    ORDER BY strength DESC, suggestion LIMIT 5
""")

### Why the depth is always bounded

Count the paths that start at one person and follow any relationship in either direction,
as the upper bound rises.

In [ ]:
growth = run(driver, """
    MATCH p = (:Person {name: 'Keanu Reeves'})-[]-{1,6}(x)
    RETURN length(p) AS hops, count(*) AS paths ORDER BY hops
""")
for row in growth:
    print(row)


Six hops from one actor already means tens of thousands of paths in a graph of 171 nodes.
On a large, densely connected graph an unbounded pattern enumerates more paths than it can
ever finish listing. **Always state an upper bound.**

The ratio alternates, small then large. Almost every relationship joins a person to a film,
so a path alternates between the two, and people have few films while films have many
people. The degree that matters at each step is not the typical node's, either: a path
reaches well-connected nodes more often, so the expected number of ways onward from a node
of a given kind is its **excess degree**, E[d²]/E[d] − 1. Over two steps, person to film to
person, the growth is roughly the product of the two.

---
## Part 5 — Neo4j from Python

`graph_db.run()` is a few lines around the official driver. This is the driver on its own,
which is what you will use in your own code.

In [ ]:
records, summary, keys = driver.execute_query(
    "MATCH (m:Movie {title: $title})<-[:ACTED_IN]-(p:Person) RETURN p.name AS actor ORDER BY actor",
    title="The Matrix",
)
for record in records:
    print(record["actor"])
print(f"columns {keys}, first row after {summary.result_available_after} ms")

`$title` is a **parameter**. Its value travels separately from the query text, so nothing in
the value can change what the query means. Now build a query by pasting the value into the
string instead, for a film that is in the graph:

In [ ]:
title = "One Flew Over the Cuckoo's Nest"      # an ordinary title that contains an apostrophe
query = "MATCH (m:Movie {title: '" + title + "'})<-[:ACTED_IN]-(p:Person) RETURN p.name AS actor"
print(query)
try:
    driver.execute_query(query)
except Exception as exc:
    print("\n" + type(exc).__name__, "--", str(exc).splitlines()[0][:110])

In [ ]:
# The parameterised version simply works.
driver.execute_query(
    "MATCH (m:Movie {title: $title})<-[:ACTED_IN]-(p:Person) RETURN p.name AS actor ORDER BY actor",
    title=title,
).records

The apostrophe broke the query. A value chosen by an attacker could instead change what the
query does, exactly as in SQL injection. From Week 5 onwards, some of the values reaching
your queries will have been written by a model. **Every value goes in as a parameter.**

---
## Part 6 — Search by meaning, then traverse

Every traversal so far started from a node named in the query. A user's question names
none. The vector index finds the starting nodes by meaning, and the graph supplies what is
connected to them.

This needs text worth embedding, and the Movies graph has little: a one-line tagline on
37 of its 38 films, averaging nine words of marketing copy. From here on the notebook uses
a graph of **this course**, in which every topic carries a description, and whose material
is the Week 3 sample corpus. Loading it first removes the Movies graph: Community edition
has one database, so the two graphs take turns in it. `python load_graph.py --movies`
brings the Movies graph back, and removes this one and its embeddings.

In [ ]:
COURSE = _root / "week4" / "course_graph.cypher"
switch_to(driver, COURSE)          # removes the Movies graph
run_script(driver, COURSE)
breakdown(driver)

You should now also see 12 `Week`, 39 `Topic`, 4 `Deliverable`, and 8 `Tool` nodes. Topics
are linked by `BUILDS_ON`, weeks `COVERS` topics, and deliverables `ASSESSES` them.

Embed each topic's `description` with the document task type, as in Week 3, store the
vectors on the nodes, and create a cosine vector index over them. Nodes that already have a
vector are skipped, so re-running this cell costs nothing.

In [ ]:
from embedding_client import EmbeddingClient
from graph_vectors import embed_nodes, vector_search, exact_search

emb = EmbeddingClient()
n = embed_nodes(driver, emb)            # label Topic, text property "description"
print(f"embedded {n} topics this time; {emb.texts_embedded} texts against the ~100/minute quota")

run(driver, """
    SHOW INDEXES YIELD name, type, state, options WHERE type = 'VECTOR'
    RETURN name, state, options.indexConfig AS config
""")

Note `vector.dimensions`, which must equal the length of the embedding model's vectors
(3072), and `vector.quantization.enabled`, which is on by default. A vector of the wrong
length is stored without any error and then silently left out of every search, which is why
`graph_vectors.py` checks the length before writing.

In [ ]:
QUESTION = "How do I find text by meaning rather than keywords?"
q_vector = emb.embed(QUESTION, task_type="RETRIEVAL_QUERY")    # a question is a query

for row in vector_search(driver, emb, QUESTION, k=3):
    print(f"{row['score']:.4f}  {row['node']['name']}")

Now the pattern the lecture built towards. The vector search finds the entry points, and
the same query continues along relationships from them.

In [ ]:
run(driver, """
    CALL db.index.vector.queryNodes('topic_embeddings', 3, $q) YIELD node AS topic, score
    MATCH (w:Week)-[:COVERS]->(topic)
    OPTIONAL MATCH (d:Deliverable)-[:ASSESSES]->(topic)
    RETURN topic.name AS topic, round(score, 4) AS score, w.number AS week,
           collect(d.name) AS assessed_by
    ORDER BY score DESC
""", q=q_vector)

Similarity chose the topics. The relationships supplied the week and the deliverable, which
no topic's description mentions. The traversal can go further: from the topic a question
lands on, to everything that topic builds on, and the week that teaches each.

In [ ]:
run(driver, """
    CALL db.index.vector.queryNodes('topic_embeddings', 1, $q) YIELD node AS topic, score
    MATCH (topic)-[:BUILDS_ON]->{1,4}(first:Topic)<-[:COVERS]-(w:Week)
    RETURN topic.name AS found, round(score, 4) AS score,
           collect(DISTINCT first.name + ' (week ' + w.number + ')') AS learn_first
""", q=q_vector)

The embedding model is multilingual, so a question in Ukrainian finds the same topics:

In [ ]:
for row in vector_search(driver, emb, "Як знайти текст за змістом, а не за ключовими словами?", k=3):
    print(f"{row['score']:.4f}  {row['node']['name']}")

### Check 1: the score is not Chroma's number

Compute the cosine similarity yourself and compare it with the score Neo4j reports.

In [ ]:
reported = run(driver, """
    CALL db.index.vector.queryNodes('topic_embeddings', 1, $q) YIELD node, score
    RETURN node.name AS name, node.embedding AS vector, score
""", q=q_vector)[0]

cos = EmbeddingClient.cosine(q_vector, reported["vector"])
print(f"top result                 {reported['name']}")
print(f"cosine, computed here      {cos:.6f}")
print(f"score Neo4j reported       {reported['score']:.6f}")

The two numbers differ. Neo4j rescales the cosine so that the score lies between 0 and 1.
Work out the formula, write it in the cell below, and check it against the numbers above.
It should agree to about three decimal places; the remaining difference is quantisation,
which stores the vectors in the index at reduced precision.

In [ ]:
my_score = None          # replace None with your formula, written in terms of cos
print(my_score, "vs", round(reported["score"], 6))

### Check 2: a filter after the search can only remove

In [ ]:
# Ask for the 3 nearest topics, then keep only those that Week 4 covers.
few = run(driver, """
    CALL db.index.vector.queryNodes('topic_embeddings', 3, $q) YIELD node, score
    MATCH (:Week {number: 4})-[:COVERS]->(node)
    RETURN node.name AS topic, round(score, 4) AS score
""", q=q_vector)
print(len(few), "row(s):", few)

# Ask for more than you need, filter, then cut down.
enough = run(driver, """
    CALL db.index.vector.queryNodes('topic_embeddings', 20, $q) YIELD node, score
    MATCH (:Week {number: 4})-[:COVERS]->(node)
    RETURN node.name AS topic, round(score, 4) AS score
    ORDER BY score DESC LIMIT 3
""", q=q_vector)
print(len(enough), "row(s):", enough)

The search found its 3 nearest topics first, and the `MATCH` then discarded the ones Week 4
does not cover. On this version a filter cannot reach inside the search. Current Neo4j
releases add a `SEARCH` clause that filters during the search, but it belongs to Cypher 25
and does not run on 5.26.

### ✍️ Your notes

**Your formula for the score, in terms of cos:**

**The same score, written in terms of Chroma's distance d = 1 − cos:**

---
## Part 7 — Answering from the graph

Week 3 ended by handing retrieved chunks to the model. This part does the same with the
graph. The question is the one the lecture opened with:

> *Does Lab 3 depend on anything taught in Week 3?*

The steps are Week 3's, with a different retriever: query the graph, turn the rows it
returns into numbered sources, and send them to the model with the same grounded
instruction. This part makes three generation calls, well within the limit of six per
minute, and embeds two questions.

In [ ]:
from context_lab import ask
from vector_store import VectorStore

# Week 3's grounded instruction, unchanged.
SYSTEM = (
    "You answer questions using only the sources you are given. If the sources do "
    "not answer the question, reply exactly: not stated. Never fill a gap from your "
    "own knowledge."
)
TASK = "Answer the question using only the SOURCES above."
FORMAT = "Two sentences maximum. Put the source number in square brackets after each claim."

QUESTION = "Does Lab 3 depend on anything taught in Week 3?"

# First, the Week 3 system: the three nearest chunks of the course material.
course_chunks = VectorStore(embedder=emb, name="week3_chunks")
if course_chunks.count() == 0:
    print("The sample course corpus is not in Chroma. Build it from week3/ with\n"
          "    python ingest.py corpus/ --name week3_chunks\n"
          "(about 180 texts, two minutes of embedding quota), or skip this cell:\n"
          "the lecture's result was: not stated.")
else:
    rows = course_chunks.search(QUESTION, n_results=3)
    ask(QUESTION, system=SYSTEM, task=TASK, fmt=FORMAT,
        sources=[r["chunk"] for r in rows], show_prompt=False, label="chunks as sources")

No passage of the course material states that connection, so a grounded answer from
chunks is *not stated*. The graph does hold it, as a chain of separate facts. The query
below follows every chain from Lab 3 to Week 3: a topic Lab 3 assesses, then zero or more
`BUILDS_ON` steps, then a topic Week 3 covers. The `{0,6}` allows zero steps, for a topic
that Week 3 covers directly. Each relationship on the chains is returned once, as one row.

In [ ]:
facts_query = """
    MATCH p = (:Deliverable {name: 'Lab 3'})-[:ASSESSES]->(:Topic)
              -[:BUILDS_ON]->{0,6}(:Topic)<-[:COVERS]-(:Week {number: 3})
    UNWIND relationships(p) AS r
    WITH DISTINCT startNode(r) AS a, type(r) AS rel, endNode(r) AS b
    RETURN coalesce(a.name, 'Week ' + toString(a.number)) AS a, rel, b.name AS b
    ORDER BY rel, a, b
"""
rows = run(driver, facts_query)
rows

The model reads text, not rows. Each row becomes one sentence, numbered as a source in
the same way Week 3 numbered chunks.

In [ ]:
VERB = {"ASSESSES": "assesses", "BUILDS_ON": "builds on", "COVERS": "covers",
        "DUE_IN": "is due in", "USES": "uses", "RUNS_IN": "runs in"}

def as_sentences(rows):
    """One sentence per relationship row: {'a', 'rel', 'b'} -> 'a verb b.'"""
    return [f"{r['a']} {VERB[r['rel']]} {r['b']}." for r in rows]

facts = as_sentences(rows)
for i, fact in enumerate(facts, 1):
    print(f"[{i}] {fact}")

result = ask(QUESTION, system=SYSTEM, task=TASK, fmt=FORMAT,
             sources=facts, show_prompt=False, label="graph facts as sources")

The instruction and the model are the same as before; only the sources changed, and the
answer changes from *not stated* to a cited *yes*. That is the whole of the graph's
contribution. The traversal connected the facts, and the model only put the connection
into words.

**Read the answer against the sources before accepting it.** The graph returns 14 facts
describing two routes, one through agent memory and one through tool safety. In three
runs recorded when this notebook was written (2026-09-27), the model answered *yes* with
citations every time, and every time followed only the route through agent memory. One
run also cited fact [3] for a claim that fact [1] supports. The runs spent between 640 and
1,024 thinking tokens on the same 244-token prompt, so your answer and your token counts
will differ from these. Which facts reach the model, and how many, is the subject of
Week 6.

### The whole pipeline: search, traverse, answer

The question above named its starting nodes, Lab 3 and Week 3. A user's question usually
names none, so the full pipeline begins with the vector index from Part 6: the search
finds the starting topic, the traversal collects what it builds on, and the model answers
from those facts. The cell prints the topic the search landed on, because a wrong
starting point is the first thing to check when an answer is wrong.

In [ ]:
QUESTION = "What do I need to know before I write an agent that calls tools?"

# 1. Search: the topic nearest in meaning to the question is the starting point.
start = vector_search(driver, emb, QUESTION, k=1)[0]
print(f"search landed on: {start['node']['name']}  (score {start['score']:.4f})\n")

# 2. Traverse: everything that topic builds on, and the week that teaches each.
rows = run(driver, """
    MATCH p = (:Topic {name: $start})-[:BUILDS_ON]->{1,6}(:Topic)<-[:COVERS]-(:Week)
    UNWIND relationships(p) AS r
    WITH DISTINCT startNode(r) AS a, type(r) AS rel, endNode(r) AS b
    RETURN coalesce(a.name, 'Week ' + toString(a.number)) AS a, rel, b.name AS b
    ORDER BY rel, a, b
""", start=start["node"]["name"])

# 3. Answer: the facts become numbered sources for the same grounded prompt.
facts = as_sentences(rows)
for i, fact in enumerate(facts, 1):
    print(f"[{i}] {fact}")

result = ask(QUESTION, system=SYSTEM, task=TASK, fmt=FORMAT,
             sources=facts, show_prompt=False, label="search, traverse, answer")

### Your turn

Ask the pipeline two questions of your own about the course: one it should answer well, and
one you expect it to answer badly. For the second, locate the fault. It lies in the
**search**, if the starting topic is wrong; in the **traversal**, if the facts the answer
needs are not among the sources; or in the **model**, if the facts are present and the
answer misuses them.

One search fault is known in advance. *"What should I learn before I use the vector index
in Neo4j?"* lands on Hybrid retrieval rather than on Vector index in Neo4j (0.8602 against
0.8509, measured 2026-09-27), because Hybrid retrieval's description, *combining vector
search and graph traversal*, is close to many questions about vectors and graphs. The
traversal from the wrong topic still reaches the right one, but it sends 46 facts to the
model instead of the dozen the question needs.

### ✍️ Your notes

**A question answered well:**

**A question answered badly, and where the fault lies:**

---
## Part 8 — How this graph was designed

Parts 6 and 7 queried a graph that someone else designed. This part
reconstructs the design, because in Week 5 you make the same decisions for your own
domain. Nothing here calls the API.

The lecture stated the method: a graph is designed backwards from the questions it must
answer. Begin with what the graph actually contains.

In [ ]:
# Every kind of relationship in the database: from which label, to which label, how many.
run(driver, """
    MATCH (a)-[r]->(b)
    RETURN labels(a)[0] AS from, type(r) AS relationship, labels(b)[0] AS to, count(*) AS n
    ORDER BY relationship
""")

In [ ]:
# The properties each label carries.
run(driver, """
    MATCH (n) UNWIND keys(n) AS key
    RETURN labels(n)[0] AS label, collect(DISTINCT key) AS properties
    ORDER BY label
""")

These two results are the graph's schema, as observed. Neo4j Browser can draw a schema with
`CALL db.schema.visualization()`, but on this version it lists every label that has a
constraint, including `Movie` and `Person` after the Movies graph has been removed. The
queries above report only what the database contains.

### Exercise 1: one question per relationship type

For each relationship type, write the question it exists to answer, in the words a student
or a lecturer would use. `BUILDS_ON` is completed as an example. A relationship type for
which no question can be stated should not be in the graph.

### Exercise 2: node or property

A week could have been a property on each topic, `week: 3`, instead of a `Week` node. A
deliverable's weight could have been a node instead of a property. Justify each of the
decisions actually made, in one sentence each, using your answers to Exercise 1.

### ✍️ Your notes

| Relationship | The question it answers |
|---|---|
| `BUILDS_ON` | What must I understand before this topic, and what depends on it? |
| `COVERS` | |
| `ASSESSES` | |
| `DUE_IN` | |
| `USES` | |
| `RUNS_IN` | |

**Why a week is a node:**

**Why a weight is a property:**

### Exercise 3: a question this graph cannot answer

> *Which passages of the course material discuss graph traversal?*

The graph records that the topic exists, which week covers it, and what it builds on. It
holds no text beyond one description per topic, so it cannot point to a passage. Design,
on paper, what would have to be added. Do not implement it; that is the work of Weeks 5
and 6. State:

1. the new node labels, and the properties each carries;
2. the new relationship types, with their direction;
3. for each new relationship, **where it would come from**.

The third point matters most. Some relationships follow mechanically from ingestion:
`ingest.py` already records which document each chunk came from. Others can be produced
only by reading the text, to decide whether a passage discusses a topic. That is the
entity extraction of Week 5.

**Where the course graph's own relationships come from.** The course graph was written by
hand from the syllabus. `COVERS`, `DUE_IN`, and `ASSESSES` restate facts the syllabus
states. Most of the 61 `BUILDS_ON` relationships are editorial judgements: no passage of
the Week 3 sample corpus states, for example, that tool safety builds on
retrieval-augmented generation. A relationship extracted from a passage can be cited in
an answer; a curated relationship can only be trusted. If you complete Part 9, you will
see that the chunks it copies into Neo4j arrive with no relationships at all.

### ✍️ Your notes

**New node labels and their properties:**

**New relationship types, with direction:**

**Where each new relationship would come from:**

---
## Part 9 (optional) — One store or two, measured

The Week 3 lecture claimed that most vector stores implement the same kind of index, so
the choice between them is operational rather than mathematical. Test that claim on your
own corpus.

The cell below copies your Week 3 chunks from Chroma into Neo4j **with their vectors**, so
no chunk is embedded again. Both stores then hold identical vectors. Each question below is
embedded once and the same vector is sent to both stores, so the only difference left is
the store itself.

In [ ]:
from graph_vectors import copy_chunks_from_chroma
from vector_store import VectorStore

COLLECTION = "chunks"      # the name your ingest.py used; the sample course corpus is "week3_chunks"

copied = copy_chunks_from_chroma(driver, COLLECTION)
store = VectorStore(embedder=emb, name=COLLECTION)
print(f"{copied} chunks copied; Chroma holds {store.count()}")

Paste your gold set from Week 3. The reference for each question is the **exact** top k:
`exact_search()` compares the question with every chunk, without using an index. That is
the answer an index approximates.

In [ ]:
GOLD = [
    {"query": "What is Lab 2 worth?", "expect": "25%"},
    # ... paste your own ten from Week 3
]
K = 5

def ids_from_neo4j(vector, k):
    rows = run(driver, """
        CALL db.index.vector.queryNodes('chunk_embeddings', $k, $v) YIELD node, score
        RETURN node.id AS id ORDER BY score DESC LIMIT $K
    """, k=k, v=vector, K=K)
    return [r["id"] for r in rows]

totals = {"Chroma": 0.0, f"Neo4j k={K}": 0.0, "Neo4j k=50, best 5": 0.0}
print(f"{'question':<46}" + "".join(f"{name:>20}" for name in totals))
for item in GOLD:
    v = emb.embed(item["query"], task_type="RETRIEVAL_QUERY")
    exact = [r["node"]["id"] for r in exact_search(driver, v, k=K)]
    found = {
        "Chroma": [r["id"] for r in store.search_by_vector(v, n_results=K)],
        f"Neo4j k={K}": ids_from_neo4j(v, K),
        "Neo4j k=50, best 5": ids_from_neo4j(v, 50),
    }
    share = {name: len(set(ids) & set(exact)) / K for name, ids in found.items()}
    for name in totals:
        totals[name] += share[name]
    print(f"{item['query'][:44]:<46}" + "".join(f"{share[name]:>20.1f}" for name in totals))

print(f"\nmean share of the exact top {K} found:")
for name, total in totals.items():
    print(f"  {name:<20} {total / len(GOLD):.3f}")

Measured on the course's own 183-chunk sample corpus (2026-09-27, eight questions), Neo4j
asked for 5 found 85% of the exact top 5 in one build of its index and 87.5% in another;
asked for 50 then cut to 5, it found 97.5% both times. Chroma returned the exact top 5 on
every question. **The mathematics is the same in both stores; the approximation is not**,
and it even varies between two builds of the same index. How closely an index approximates the exact answer, and
what you must do to bring it closer, is part of the operational choice between stores.
Your own figures will differ: record them.

### ✍️ Your notes

**Your three figures:**

**For Lab 2, one store or two, and why:**

---
## Before you close this

- `docker compose up` works on **your own** machine, and the image is pulled.
- The Movies graph loads twice with the same counts.
- One search-then-traverse query works from Python, on the course graph.
- One question answered by the model from graph facts, with the answer checked against
  its sources.

**Lab 1** is due at the end of this week. Report retrieval quality as the hit count from
your Week 3 notebook's `hits_at_k`.

**For Week 5**, bring the entities and the three or more relationship types from your
approved project proposal, and three to five questions about your domain whose answers
depend on connections between entities rather than on any single passage. Week 5 designs
a graph of your own domain from those questions.

In [ ]:
print(f"texts embedded this session: {emb.texts_embedded}")
driver.close()